# TokenSHAP vs Standard SHAP Token Attribution Benchmark

This notebook compares token-level attributions produced by [TokenSHAP](https://github.com/GenAISHAP/TokenSHAP) (Monte Carlo Shapley value estimation) against baseline SHAP token attributions on prompt sequences from `analogies.txt`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IamKrill1n/circuit_tracer_mod/blob/main/tokenshap_demo.ipynb)

## 1. Environment & Dependency Setup
Install required dependencies for Google Colab or local execution.

In [ ]:
# Install TokenSHAP and baseline dependencies
!pip install -q git+https://github.com/GenAISHAP/TokenSHAP.git transformers torch shap entmax matplotlib pandas seaborn huggingface_hub

## 1b. Hugging Face Authentication
Gated models such as `google/gemma-2-2b` or `meta-llama/Llama-3.2-3B-Instruct` require accepting model terms on Hugging Face and logging in with an HF Token.

In [ ]:
import os
from huggingface_hub import login

# Check environment variables or Google Colab Secrets (HF_TOKEN / HUGGINGFACE_API_KEY)
hf_token = os.environ.get("HUGGINGFACE_API_KEY") or os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_API_KEY')
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face Hub!")
else:
    print("HF_TOKEN not found in environment or Colab secrets. Opening interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()

## 2. Load Prompts Dataset
Fetch `analogies.txt` from repository if running in a standalone Google Colab session.

In [ ]:
import os
from pathlib import Path

ANALOGIES_PATH = Path("analogies.txt")
if not ANALOGIES_PATH.exists():
    print("Downloading analogies.txt from repository...")
    !wget -q https://raw.githubusercontent.com/IamKrill1n/circuit_tracer_mod/main/analogies.txt -O analogies.txt

with open("analogies.txt", "r") as f:
    sample_prompts = [line.strip() for line in f if line.strip()][:5]

print(f"Loaded {len(sample_prompts)} sample prompts:")
for idx, p in enumerate(sample_prompts, 1):
    print(f"{idx}. {p}")

## 3. Initialize Device & Models
Initialize CUDA device and set up model name.

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Note: Change to an ungated model like 'Qwen/Qwen2.5-1.5B-Instruct' or 'gpt2' if you don't have Gemma permissions.
MODEL_NAME = "google/gemma-2-2b"

## 4. Run TokenSHAP
Compute Shapley values using TokenSHAP's Monte Carlo estimation engine.

In [ ]:
# TokenSHAP sub-module imports
from token_shap import TokenSHAP
from token_shap.base import LocalModel
from token_shap.token_shap import StringSplitter

print(f"Loading TokenSHAP model ({MODEL_NAME})...")
ts_model = LocalModel(MODEL_NAME, device=device)
token_shap_evaluator = TokenSHAP(ts_model, StringSplitter())

def compute_tokenshap(prompt):
    df_ts = token_shap_evaluator.analyze(prompt, sampling_ratio=0.0, print_highlight_text=False)
    return df_ts

## 5. Run Baseline SHAP
Compute baseline SHAP token attributions using standard Hugging Face `shap.Explainer`.

In [ ]:
import shap
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading baseline SHAP model ({MODEL_NAME})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device)

explainer = shap.Explainer(hf_model, tokenizer)

def compute_baseline_shap(prompt):
    shap_values = explainer([prompt])
    tokens = shap_values.data[0]
    vals = shap_values.values[0]
    if len(vals.shape) > 1:
        vals = vals.sum(axis=-1)
    return pd.DataFrame({"Token": tokens, "SHAP Value": vals})

## 6. Visual Side-by-Side Heatmaps
Compare token importance heatmaps and top token rankings side-by-side.

In [ ]:
def display_side_by_side_heatmaps(prompt, df_ts, df_base):
    fig, axes = plt.subplots(1, 2, figsize=(18, 4.5))
    
    # TokenSHAP heatmap
    ts_tokens = df_ts.iloc[:, 0].astype(str).values
    ts_scores = df_ts.iloc[:, 1].values.astype(float)
    sns.heatmap([ts_scores], xticklabels=ts_tokens, yticklabels=["TokenSHAP"],
                cmap="coolwarm", annot=True, fmt=".3f", cbar=True, ax=axes[0])
    axes[0].set_title(f"TokenSHAP: {prompt[:35]}...")
    axes[0].tick_params(axis='x', rotation=45)
    
    # Baseline SHAP heatmap
    b_tokens = df_base["Token"].astype(str).values
    b_scores = df_base["SHAP Value"].values.astype(float)
    sns.heatmap([b_scores], xticklabels=b_tokens, yticklabels=["Baseline SHAP"],
                cmap="coolwarm", annot=True, fmt=".3f", cbar=True, ax=axes[1])
    axes[1].set_title(f"Baseline SHAP: {prompt[:35]}...")
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

for prompt in sample_prompts:
    print(f"\nAnalyzing prompt: {prompt}")
    try:
        df_ts = compute_tokenshap(prompt)
        df_base = compute_baseline_shap(prompt)
        display_side_by_side_heatmaps(prompt, df_ts, df_base)
    except Exception as err:
        print(f"Error processing prompt '{prompt}': {err}")